In [2]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
import os
from scipy.sparse import csr_matrix, lil_matrix, eye, coo_matrix
import scipy.sparse as sp
from numpy.linalg import solve
import pickle
import pyomo.environ as pyo
from pyomo.contrib.fbbt import interval
import math, re

import copy
print('done') 

done


In [7]:
nsamples = 9

# === Initialization ===
case_name = 'pglib_opf_case300_ieee' #
case_path = f"M:/OneDrive - KFUPM/ISE 712 PhD. Dissertation/code/excel_outputs/{case_name}.xlsx"
# case_path = f"D:/mdz/OneDrive - KFUPM/ISE 712 PhD. Dissertation/code/excel_outputs/{case_name}.xlsx"
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

# Force these DataFrames to be float64 to avoid dtype warnings
case['bus'] = case['bus'].astype(float)
case['gen'] = case['gen'].astype(float)
case['branch'] = case['branch'].astype(float)

# Force bus IDs to be clean integers (removes string/float mismatches from Excel)
case['bus']['bus_i'] = case['bus']['bus_i'].astype(int)
case['branch']['bus_i'] = case['branch']['bus_i'].astype(int)
case['branch']['bus_j'] = case['branch']['bus_j'].astype(int)
case['gen']['bus_i'] = case['gen']['bus_i'].astype(int)

bus_to_idx = {bus: i for i, bus in enumerate(case['bus'].bus_i.values)}
bus_idx = [bus_to_idx[bus] for bus in case['bus'].bus_i.values]
case['bus'].bus_i = case['bus'].bus_i.replace(bus_to_idx)
case['gen'].bus_i = case['gen'].bus_i.replace(bus_to_idx)
baseMVA = int(case['baseMVA']['baseMVA'][0])

case['branch'].bus_i = case['branch'].bus_i.replace(bus_to_idx)
case['branch'].bus_j = case['branch'].bus_j.replace(bus_to_idx)


nbus = case['bus'].shape[0]
ngen = case['gen'].shape[0]
nbranch = case['branch'].shape[0]

# zzz comment this after
case['branch'].rateA = np.array([9900 for _ in range(nbranch)])
print('nbranch', nbranch)

nbranch 411


In [8]:
# === Scale Data to Per-Unit (pu) ===
# This replaces tx_utils.scale_ModelData_to_pu(md, inplace=True)

# 1. Scale Bus Loads and Shunts
case['bus']['Pd'] = case['bus']['Pd'] / baseMVA
case['bus']['Qd'] = case['bus']['Qd'] / baseMVA
case['bus']['Gs'] = case['bus']['Gs'] / baseMVA
case['bus']['Bs'] = case['bus']['Bs'] / baseMVA

# 2. Scale Generator Power Limits
case['gen']['Pmax'] = case['gen']['Pmax'] / baseMVA
case['gen']['Pmin'] = case['gen']['Pmin'] / baseMVA
case['gen']['Qmax'] = case['gen']['Qmax'] / baseMVA
case['gen']['Qmin'] = case['gen']['Qmin'] / baseMVA

# 3. Scale Branch Thermal Limits
# MATPOWER uses 0 to indicate "unlimited". We will scale it, and handle the 0s later in the constraints.
case['branch']['rateA'] = case['branch']['rateA'] / baseMVA
case['branch']['rateB'] = case['branch']['rateB'] / baseMVA
case['branch']['rateC'] = case['branch']['rateC'] / baseMVA

# # # === Pre-Processing: Remove Out-of-Service Elements ===
# # Keep only elements where 'status' is 1 (online)
# case['gen'] = case['gen'][case['gen']['status'] == 1]
# case['branch'] = case['branch'][case['branch']['status'] == 1]

# === Pre-Processing: Remove Out-of-Service Elements ===
# 1. Create a mask for online generators
active_gens = case['gen']['status'] == 1

# 2. Filter BOTH tables so they stay perfectly aligned
case['gen'] = case['gen'][active_gens]
case['gencost'] = case['gencost'][active_gens]

case['branch'] = case['branch'][case['branch']['status'] == 1]

In [9]:
def makeYbus(baseMVA, bus, branch):
    """
    Builds the bus admittance matrix and branch admittance matrices.
    
    Args:
        baseMVA (float): System base MVA.
        bus (pd.DataFrame): Bus data dataframe.
        branch (pd.DataFrame): Branch data dataframe.
        
    Returns:
        Ybus (scipy.sparse.csr_matrix): Bus admittance matrix (nb x nb).
        Yf (scipy.sparse.csr_matrix): From-branch admittance matrix (nl x nb).
        Yt (scipy.sparse.csr_matrix): To-branch admittance matrix (nl x nb).
    """
    nb = len(bus)
    nl = len(branch)

    # 1. Ensure 0-based indices for sparse matrix coordinates
    # Maps arbitrary bus IDs to continuous 0-based integers (0 to nb-1)
    bus_pos = {b_id: i for i, b_id in enumerate(bus['bus_i'].values)}
    
    f = np.array([bus_pos[b] for b in branch['bus_i'].values])
    t = np.array([bus_pos[b] for b in branch['bus_j'].values])
    
    # 2. Extract branch parameters (safely handling blank Excel cells as NaNs)
    stat = branch['status'].fillna(1.0).values if 'status' in branch.columns else np.ones(nl)
    r = branch['r'].values
    x = branch['x'].values
    b_ch = branch['b'].fillna(0.0).values if 'b' in branch.columns else np.zeros(nl)
    
    # Tap ratio logic (Blanks -> 1.0, and 0.0 -> 1.0)
    tap = branch['ratio'].fillna(1.0).values.copy() if 'ratio' in branch.columns else np.ones(nl)
    tap[tap == 0.0] = 1.0  
    
    # Phase shifters (Blanks -> 0.0)
    shift = branch['angle'].fillna(0.0).values if 'angle' in branch.columns else np.zeros(nl)
    tap_complex = tap * np.exp(1j * np.pi / 180.0 * shift)
    
    # 3. Compute Series Admittance and Line Charging
    Ys = stat / (r + 1j * x)
    Bc = stat * b_ch
    
    # 4. Compute Branch Admittance Matrix Elements
    Ytt = Ys + 1j * Bc / 2.0
    Yff = Ytt / (tap_complex * np.conj(tap_complex))
    Yft = -Ys / np.conj(tap_complex)
    Ytf = -Ys / tap_complex
    
    # 5. Compute Shunt Admittance at Buses
    Gs = bus['Gs'].values if 'Gs' in bus.columns else np.zeros(nb)
    Bs = bus['Bs'].values if 'Bs' in bus.columns else np.zeros(nb)
    Ysh = (Gs + 1j * Bs) / baseMVA
    
    # 6. Build Yf and Yt (Branch Admittance Matrices)
    # Rows: 0 to nl-1 (twice, once for 'from' bus, once for 'to' bus)
    i_idx = np.concatenate([np.arange(nl), np.arange(nl)])
    j_idx = np.concatenate([f, t])
    
    data_f = np.concatenate([Yff, Yft])
    Yf = coo_matrix((data_f, (i_idx, j_idx)), shape=(nl, nb)).tocsr()
    
    data_t = np.concatenate([Ytf, Ytt])
    Yt = coo_matrix((data_t, (i_idx, j_idx)), shape=(nl, nb)).tocsr()
    
    # 7. Build Ybus (System Admittance Matrix)
    # row coordinates: f, f, t, t, 0..nb-1
    # col coordinates: f, t, f, t, 0..nb-1
    row_ybus = np.concatenate([f, f, t, t, np.arange(nb)])
    col_ybus = np.concatenate([f, t, f, t, np.arange(nb)])
    data_ybus = np.concatenate([Yff, Yft, Ytf, Ytt, Ysh])
    
    Ybus = coo_matrix((data_ybus, (row_ybus, col_ybus)), shape=(nb, nb)).tocsr()
    
    return Ybus, Yf, Yt

def calculate_y_matrix(rs, xs, bs, tau, shift):
    """ Standard Pi-model Y-matrix calculation used by EGRET """
    import math
    # Impedance z = r + jx
    # Admittance y = 1/z = (r - jx) / (r^2 + x^2)
    zm_sq = rs**2 + xs**2
    gs = rs / zm_sq
    bs_prime = -xs / zm_sq
    
    # Phase shift in radians
    phi = math.radians(shift)
    cos_phi = math.cos(phi)
    sin_phi = math.sin(phi)
    
    # Pre-calculating Y-matrix elements for rectangular current calc
    y = {}
    # From-side current real part (ifr) coefficients
    y['ifr', 'vfr'] = (gs + 0.0) / tau**2
    y['ifr', 'vfj'] = (-bs_prime - bs/2.0) / tau**2
    y['ifr', 'vtr'] = (-gs * cos_phi + bs_prime * sin_phi) / tau
    y['ifr', 'vtj'] = (gs * sin_phi + bs_prime * cos_phi) / tau
    
    # From-side current imag part (ifj) coefficients
    y['ifj', 'vfr'] = (bs_prime + bs/2.0) / tau**2
    y['ifj', 'vfj'] = (gs + 0.0) / tau**2
    y['ifj', 'vtr'] = (-gs * sin_phi - bs_prime * cos_phi) / tau
    y['ifj', 'vtj'] = (-gs * cos_phi + bs_prime * sin_phi) / tau
    
    # To-side current real part (itr) coefficients
    y['itr', 'vfr'] = (-gs * cos_phi - bs_prime * sin_phi) / tau
    y['itr', 'vfj'] = (-gs * sin_phi + bs_prime * cos_phi) / tau
    y['itr', 'vtr'] = gs
    y['itr', 'vtj'] = (-bs_prime - bs/2.0)
    
    # To-side current imag part (itj) coefficients
    y['itj', 'vfr'] = (gs * sin_phi - bs_prime * cos_phi) / tau
    y['itj', 'vfj'] = (-gs * cos_phi - bs_prime * sin_phi) / tau
    y['itj', 'vtr'] = (bs_prime + bs/2.0)
    y['itj', 'vtj'] = gs
    
    return y


In [10]:
# === 0. Initialize the Pyomo Model ===
model = pyo.ConcreteModel()

# === 1. Create the Pyomo Sets (Equivalent to *_attrs['names']) ===
bus_indices = case['bus'].index.tolist()
gen_indices = case['gen'].index.tolist()
branch_indices = case['branch'].index.tolist()

model.Buses = pyo.Set(initialize=bus_indices, ordered=True)
model.Gens = pyo.Set(initialize=gen_indices, ordered=True)
model.Branches = pyo.Set(initialize=branch_indices, ordered=True)

# === 2. Convert DataFrames to Dictionaries (Equivalent to md.elements(...)) ===
# This creates dictionaries where the keys are the indices (e.g., bus IDs) 
# and the values are dictionaries of that row's data.
buses = case['bus'].to_dict(orient='index')
gens = case['gen'].to_dict(orient='index')
branches = case['branch'].to_dict(orient='index')

# === 3. Map Branches to Buses (Inlet/Outlet) ===

# Because we set the index above, bus_indices now safely contains 
# your original MATPOWER IDs (31, 89, 228, etc.)
inlet_branches_by_bus = {i: [] for i in bus_indices}
outlet_branches_by_bus = {i: [] for i in bus_indices}

for row in case['branch'].itertuples():
    branch_idx = row.Index
    
    # row.bus_i is the "from" bus
    outlet_branches_by_bus[row.bus_i].append(branch_idx)
    
    # row.bus_j is the "to" bus
    inlet_branches_by_bus[row.bus_j].append(branch_idx)
    
# === 4. Map Generators to Buses ===
# Initialize dictionary with empty lists for every bus
gens_by_bus = {int(i): [] for i in case['bus']['bus_i'].values}
# gens_by_bus = {i: [] for i in bus_indices}

# Iterate through the generator DataFrame to populate the lists
for row in case['gen'].itertuples():
    gen_idx = row.Index
    # row.bus_i is the bus to which this generator is connected
    gens_by_bus[row.bus_i].append(gen_idx)

# === 5. Find Unique Bus Pairs (for parallel lines) ===
# Combine 'from' and 'to' buses into a list of tuples
all_bus_pairs = list(zip(case['branch']['bus_i'], case['branch']['bus_j']))

# Use dict.fromkeys() to remove duplicates while preserving insertion order.
# (In modern Python, standard dictionaries maintain insertion order, 
# acting exactly like EGRET's OrderedDict approach).
unique_bus_pairs = list(dict.fromkeys(all_bus_pairs))

# Optional: Create a Pyomo Set for these pairs
model.UniqueBusPairs = pyo.Set(initialize=unique_bus_pairs, dimen=2, ordered=True)    

# === 6. Extract and Fix Bus Loads ===

# 1. Map the bus index to the Pd and Qd columns directly
# (Using .itertuples() for speed, as usual)
bus_p_loads = {row.Index: row.Pd for row in case['bus'].itertuples()}
bus_q_loads = {row.Index: row.Qd for row in case['bus'].itertuples()}

# 2. Declare the load variables on the Pyomo model
model.pl = pyo.Var(model.Buses, initialize=bus_p_loads)
model.ql = pyo.Var(model.Buses, initialize=bus_q_loads)

# 3. Fix the variables so the solver treats them as constants
model.pl.fix()
model.ql.fix()

# === 7. Extract Fixed Shunts ===

# Extract Conductance (Gs) and Susceptance (Bs) directly from the DataFrame
# (Using .itertuples() for speed)
bus_gs_fixed_shunts = {row.Index: row.Gs for row in case['bus'].itertuples()}
bus_bs_fixed_shunts = {row.Index: row.Bs for row in case['bus'].itertuples()}

# === 8. Declare Auxiliary Variable: Voltage Magnitude Squared (vmsq) ===

# 1. Create the initialization dictionary (Vm^2)
# (Using .itertuples() for speed)
vmsq_init = {row.Index: row.Vm**2 for row in case['bus'].itertuples()}

# 2. Define the bounds rule (Vmin^2, Vmax^2)
def vmsq_bounds_rule(m, b):
    vmin = case['bus'].at[b, 'Vmin']
    vmax = case['bus'].at[b, 'Vmax']
    return (vmin**2, vmax**2)

# 3. Declare the Pyomo variable natively
model.vmsq = pyo.Var(model.Buses, domain=pyo.Reals, initialize=vmsq_init, bounds=vmsq_bounds_rule)


# === 9. Declare Auxiliary Variable 'c': vi * vj * cos(theta_i - theta_j) ===

# 1. Create a lookup dictionary for angle limits by bus pair
# In MATPOWER, if angmin and angmax are both 0, it means "unbounded" (-360 to 360).
branch_limits = {}
for row in case['branch'].itertuples():
    if row.angmin == 0.0 and row.angmax == 0.0:
        angmin, angmax = -360.0, 360.0
    else:
        angmin, angmax = row.angmin, row.angmax
    
    # If there are parallel lines, this will overwrite and grab the last one.
    # This is safe because parallel lines between the same buses share the same physical angle difference.
    branch_limits[(row.bus_i, row.bus_j)] = (angmin, angmax)

# 2. Define the exact bounds using interval arithmetic
def c_bounds_rule(m, i, j):
    # Get angle limits in radians
    angmin_deg, angmax_deg = branch_limits[(i, j)]
    theta_bounds = (math.radians(angmin_deg), math.radians(angmax_deg))
    
    # Get voltage limits
    vf_bounds = (case['bus'].at[i, 'Vmin'], case['bus'].at[i, 'Vmax'])
    vt_bounds = (case['bus'].at[j, 'Vmin'], case['bus'].at[j, 'Vmax'])
    
    # Calculate bounds: c_bounds = v_i * v_j * cos(theta_i - theta_j)
    # The `*` unpacks the tuple into two arguments for the interval functions
    c_bnds = interval.cos(*theta_bounds)
    c_bnds = interval.mul(*vf_bounds, *c_bnds)
    c_bnds = interval.mul(*vt_bounds, *c_bnds)
    
    return c_bnds

# 3. Declare the variable natively on the UniqueBusPairs set
model.c = pyo.Var(model.UniqueBusPairs, domain=pyo.Reals, initialize=1.0, bounds=c_bounds_rule)

# === 10. Declare Auxiliary Variable 's': vi * vj * sin(theta_i - theta_j) ===

def s_bounds_rule(m, i, j):
    # Get angle limits in radians (reusing the branch_limits dictionary from the 'c' step)
    angmin_deg, angmax_deg = branch_limits[(i, j)]
    theta_bounds = (math.radians(angmin_deg), math.radians(angmax_deg))
    
    # Get voltage limits from the bus DataFrame
    vf_bounds = (case['bus'].at[i, 'Vmin'], case['bus'].at[i, 'Vmax'])
    vt_bounds = (case['bus'].at[j, 'Vmin'], case['bus'].at[j, 'Vmax'])
    
    # Calculate bounds: s_bounds = v_i * v_j * sin(theta_i - theta_j)
    s_bnds = interval.sin(*theta_bounds)
    s_bnds = interval.mul(*vf_bounds, *s_bnds)
    s_bnds = interval.mul(*vt_bounds, *s_bnds)
    
    return s_bnds

# Declare the 's' variable natively on the UniqueBusPairs set
model.s = pyo.Var(model.UniqueBusPairs, domain=pyo.Reals, initialize=0.0, bounds=s_bounds_rule)

# === 11. Declare Generator Real and Reactive Power (Pg, Qg) ===

# 1. Rules for Active Power (Pg)
def pg_bounds_rule(m, g):
    return (case['gen'].at[g, 'Pmin'], case['gen'].at[g, 'Pmax'])

def pg_init_rule(m, g):
    # Initialize at the midpoint of the bounds
    return (case['gen'].at[g, 'Pmin'] + case['gen'].at[g, 'Pmax']) / 2.0

model.pg = pyo.Var(model.Gens, domain=pyo.Reals, bounds=pg_bounds_rule, initialize=pg_init_rule)


# 2. Rules for Reactive Power (Qg)
def qg_bounds_rule(m, g):
    return (case['gen'].at[g, 'Qmin'], case['gen'].at[g, 'Qmax'])

def qg_init_rule(m, g):
    # Initialize at the midpoint of the bounds
    return (case['gen'].at[g, 'Qmin'] + case['gen'].at[g, 'Qmax']) / 2.0

model.qg = pyo.Var(model.Gens, domain=pyo.Reals, bounds=qg_bounds_rule, initialize=qg_init_rule)

# === 12. Setup Branch Power Flow Limits ===

# 1. Rectangular Voltage Initializations 
# (Note: Vm is in pu, Va is in degrees in MATPOWER)
vr_init = {row.Index: row.Vm * math.cos(math.radians(row.Va)) for row in case['bus'].itertuples()}
vj_init = {row.Index: row.Vm * math.sin(math.radians(row.Va)) for row in case['bus'].itertuples()}

# 2. Define a unified bounds rule for pf, pt, qf, qt based on rateA
def branch_flow_bounds_rule(m, b):
    # Remember: We already divided rateA by baseMVA earlier!
    rateA = case['branch'].at[b, 'rateA']
    
    # In MATPOWER, 0 means unlimited capacity
    if rateA == 0.0:
        return (None, None)
    else:
        return (-rateA, rateA)

# (We don't need to create pf_bounds, pt_bounds dictionaries here, 
# because we will just pass `bounds=branch_flow_bounds_rule` directly 
# into pyo.Var() in the next step!)

# === 13. Final Branch Flow Initialization & Declaration (with Clipping) ===

pf_init, pt_init, qf_init, qt_init = {}, {}, {}, {}

for row in case['branch'].itertuples():
    idx = row.Index
    f_bus, t_bus = row.bus_i, row.bus_j
    
    # 1. Calculate physical flows
    tau = row.ratio if row.ratio != 0 else 1.0
    shift = row.angle
    y_mat = calculate_y_matrix(row.r, row.x, row.b, tau, shift)
    
    vfr, vfj = vr_init[f_bus], vj_init[f_bus]
    vtr, vtj = vr_init[t_bus], vj_init[t_bus]
    
    ifr = y_mat['ifr','vfr']*vfr + y_mat['ifr','vfj']*vfj + y_mat['ifr','vtr']*vtr + y_mat['ifr','vtj']*vtj
    ifj = y_mat['ifj','vfr']*vfr + y_mat['ifj','vfj']*vfj + y_mat['ifj','vtr']*vtr + y_mat['ifj','vtj']*vtj
    itr = y_mat['itr','vfr']*vfr + y_mat['itr','vfj']*vfj + y_mat['itr','vtr']*vtr + y_mat['itr','vtj']*vtj
    itj = y_mat['itj','vfr']*vfr + y_mat['itj','vfj']*vfj + y_mat['itj','vtr']*vtr + y_mat['itj','vtj']*vtj
    
    p_f = vfr * ifr + vfj * ifj
    q_f = vfj * ifr - vfr * ifj
    p_t = vtr * itr + vtj * itj
    q_t = vtj * itr - vtr * itj

    # 2. CLIP to bounds to remove W1002 warnings
    rateA = row.rateA
    if rateA > 0:
        pf_init[idx] = max(min(p_f, rateA), -rateA)
        qf_init[idx] = max(min(q_f, rateA), -rateA)
        pt_init[idx] = max(min(p_t, rateA), -rateA)
        qt_init[idx] = max(min(q_t, rateA), -rateA)
    else:
        pf_init[idx], qf_init[idx] = p_f, q_f
        pt_init[idx], qt_init[idx] = p_t, q_t

# 3. Declare variables
model.pf = pyo.Var(model.Branches, bounds=branch_flow_bounds_rule, initialize=pf_init)
model.pt = pyo.Var(model.Branches, bounds=branch_flow_bounds_rule, initialize=pt_init)
model.qf = pyo.Var(model.Branches, bounds=branch_flow_bounds_rule, initialize=qf_init)
model.qt = pyo.Var(model.Branches, bounds=branch_flow_bounds_rule, initialize=qt_init)

# === 14. Branch Power Flow Constraints ===
# This block links the flow variables (pf, pt, qf, qt) to the 
# voltage variables (vmsq, c, s) using the admittance matrix.

# --- Helper Functions (Standard conductance/susceptance) ---
def calculate_conductance(r, x):
    return r / (r**2 + x**2)

def calculate_susceptance(r, x):
    return -x / (r**2 + x**2)

# --- Define the Constraints ---
model.eq_pf_branch = pyo.Constraint(model.Branches)
model.eq_pt_branch = pyo.Constraint(model.Branches)
model.eq_qf_branch = pyo.Constraint(model.Branches)
model.eq_qt_branch = pyo.Constraint(model.Branches)

for row in case['branch'].itertuples():
    idx = row.Index
    from_bus = row.bus_i
    to_bus = row.bus_j
    
    # 1. Calculate physical parameters
    g = calculate_conductance(row.r, row.x)
    b = calculate_susceptance(row.r, row.x)
    bc = row.b # charging susceptance
    
    tau = row.ratio if row.ratio != 0 else 1.0
    shift = math.radians(row.angle)
    
    # 2. Compute Admittance Matrix Coefficients (G and B matrix elements)
    g11 = g / tau**2
    g12 = g * math.cos(shift) / tau
    g21 = g * math.sin(shift) / tau
    g22 = g

    b11 = (b + bc / 2) / tau**2
    b12 = b * math.cos(shift) / tau
    b21 = b * math.sin(shift) / tau
    b22 = b + bc / 2

    # 3. Apply the constraints to the model
    # Real Power (From side)
    model.eq_pf_branch[idx] = model.pf[idx] == \
        g11 * model.vmsq[from_bus] - \
        (g12 * model.c[(from_bus, to_bus)] + 
         g21 * model.s[(from_bus, to_bus)] + 
         b12 * model.s[(from_bus, to_bus)] - 
         b21 * model.c[(from_bus, to_bus)])

    # Real Power (To side)
    model.eq_pt_branch[idx] = model.pt[idx] == \
        g22 * model.vmsq[to_bus] - \
        (g12 * model.c[(from_bus, to_bus)] + 
         g21 * model.s[(from_bus, to_bus)] - 
         b12 * model.s[(from_bus, to_bus)] + 
         b21 * model.c[(from_bus, to_bus)])

    # Reactive Power (From side)
    model.eq_qf_branch[idx] = model.qf[idx] == \
        -b11 * model.vmsq[from_bus] + \
        (b12 * model.c[(from_bus, to_bus)] + 
         b21 * model.s[(from_bus, to_bus)] - 
         g12 * model.s[(from_bus, to_bus)] + 
         g21 * model.c[(from_bus, to_bus)])

    # Reactive Power (To side)
    model.eq_qt_branch[idx] = model.qt[idx] == \
        -b22 * model.vmsq[to_bus] + \
        (b12 * model.c[(from_bus, to_bus)] + 
         b21 * model.s[(from_bus, to_bus)] + 
         g12 * model.s[(from_bus, to_bus)] - 
         g21 * model.c[(from_bus, to_bus)])
    
# === 15. Nodal Power Balance (P & Q Balances) ===

# Initialize the constraints on the model
model.eq_p_balance = pyo.Constraint(model.Buses)
model.eq_q_balance = pyo.Constraint(model.Buses)

for b in case['bus'].bus_i.values: #model.Buses
    # --- Real Power Balance (P) ---
    # 1. Power from Generators connected to this bus
    p_expr = sum(model.pg[g] for g in gens_by_bus[b])
    
    # 2. Subtract Power Flowing OUT (pf) and OUT (pt) 
    # (EGRET uses this convention: sum of all branch terminal flows)
    p_expr -= sum(model.pf[br] for br in outlet_branches_by_bus[b])
    p_expr -= sum(model.pt[br] for br in inlet_branches_by_bus[b])
    
    # 3. Subtract Fixed Shunt Conductance (Gs * V^2)
    if bus_gs_fixed_shunts[b] != 0.0:
        p_expr -= bus_gs_fixed_shunts[b] * model.vmsq[b]
        
    # 4. Subtract Fixed Load (pl)
    # Since we fixed model.pl earlier, we subtract the variable itself
    p_expr -= model.pl[b]
    
    # Apply Constraint: Sum of Injections == 0
    model.eq_p_balance[b] = (p_expr == 0.0)


    # --- Reactive Power Balance (Q) ---
    # 1. Reactive Power from Generators
    q_expr = sum(model.qg[g] for g in gens_by_bus[b])
    
    # 2. Subtract Reactive Flows OUT (qf) and OUT (qt)
    q_expr -= sum(model.qf[br] for br in outlet_branches_by_bus[b])
    q_expr -= sum(model.qt[br] for br in inlet_branches_by_bus[b])
    
    # 3. Add Fixed Shunt Susceptance (Bs * V^2)
    # Note: Reactive power injection from a capacitor (Bs > 0) is +Bs*V^2
    if bus_bs_fixed_shunts[b] != 0.0:
        q_expr += bus_bs_fixed_shunts[b] * model.vmsq[b]
        
    # 4. Subtract Fixed Load (ql)
    q_expr -= model.ql[b]
    
    # Apply Constraint: Sum of Injections == 0
    model.eq_q_balance[b] = (q_expr == 0.0)
        
# === 16. Branch Thermal Limits (Apparent Power Limits) ===

# Initialize the inequality constraints on the model
model.ineq_sf_branch_thermal_limit = pyo.Constraint(model.Branches)
model.ineq_st_branch_thermal_limit = pyo.Constraint(model.Branches)

for row in case['branch'].itertuples():
    idx = row.Index
    s_max = row.rateA  # This is already scaled to per-unit from our early steps
    
    # In MATPOWER/PGLib, a rateA of 0.0 means the line is unlimited.
    # We only add the constraint if a limit actually exists.
    if s_max > 0.0:
        # From-side limit: P_f^2 + Q_f^2 <= S_max^2
        model.ineq_sf_branch_thermal_limit[idx] = \
            model.pf[idx]**2 + model.qf[idx]**2 <= s_max**2
        
        # To-side limit: P_t^2 + Q_t^2 <= S_max^2
        model.ineq_st_branch_thermal_limit[idx] = \
            model.pt[idx]**2 + model.qt[idx]**2 <= s_max**2
    else:
        # If rateA is 0, we can skip adding the constraint to keep the model small
        model.ineq_sf_branch_thermal_limit[idx] = pyo.Constraint.Skip
        model.ineq_st_branch_thermal_limit[idx] = pyo.Constraint.Skip        

# === 17. Angle Difference Limits (FIXED) ===

model.ineq_angle_diff_branch_lb = pyo.Constraint(model.Branches)
model.ineq_angle_diff_branch_ub = pyo.Constraint(model.Branches)

for row in case['branch'].itertuples():
    idx = row.Index
    from_bus = row.bus_i
    to_bus = row.bus_j
    
    # Catch the MATPOWER "0 means unbounded" trap!
    if row.angmin == 0.0 and row.angmax == 0.0:
        ang_min_deg, ang_max_deg = -360.0, 360.0
    else:
        ang_min_deg = row.angmin
        ang_max_deg = row.angmax
    
    # Lower bound constraint
    if ang_min_deg > -90:
        model.ineq_angle_diff_branch_lb[idx] = (
            math.tan(math.radians(ang_min_deg)) * model.c[(from_bus, to_bus)] 
            <= model.s[(from_bus, to_bus)]
        )
    else:
        model.ineq_angle_diff_branch_lb[idx] = pyo.Constraint.Skip

    # Upper bound constraint
    if ang_max_deg < 90:
        model.ineq_angle_diff_branch_ub[idx] = (
            model.s[(from_bus, to_bus)] 
            <= math.tan(math.radians(ang_max_deg)) * model.c[(from_bus, to_bus)]
        )
    else:
        model.ineq_angle_diff_branch_ub[idx] = pyo.Constraint.Skip
        
# === 18. Generator Cost Objective (Polynomial/Quadratic) ===

# 1. Pre-extract coefficients into dictionaries to avoid Pandas indexing errors
# Use .at[g, 'column'] to safely pull the exact row matching the generator ID
c2_dict = {g: float(case['gencost'].at[g, 'c2']) for g in model.Gens}
c1_dict = {g: float(case['gencost'].at[g, 'c1']) for g in model.Gens}
c0_dict = {g: float(case['gencost'].at[g, 'c0']) for g in model.Gens}

# # We use the generator names/indices as keys
# c2_dict = {g: float(case['gencost']['c2'].iloc[i]) for i, g in enumerate(model.Gens)}
# c1_dict = {g: float(case['gencost']['c1'].iloc[i]) for i, g in enumerate(model.Gens)}
# c0_dict = {g: float(case['gencost']['c0'].iloc[i]) for i, g in enumerate(model.Gens)}

# 2. Define the cost expression rule using the clean dictionaries
def pg_cost_exp_rule(m, g):
    # Scale pu to MW
    baseMVA = int(case['baseMVA']['baseMVA'][0])
    pg_mw = m.pg[g] * baseMVA
    
    # Calculate quadratic cost: c2*Pg^2 + c1*Pg + c0
    return c2_dict[g] * (pg_mw**2) + c1_dict[g] * pg_mw + c0_dict[g]

# 3. Create the Pyomo Expression
model.pg_operating_cost = pyo.Expression(model.Gens, rule=pg_cost_exp_rule)

# 4. Define and add the Objective to the model
def total_cost_rule(m):
    return sum(m.pg_operating_cost[g] for g in m.Gens)

model.obj = pyo.Objective(rule=total_cost_rule, sense=pyo.minimize)

# === RSV Step 1: Declare Rectangular Voltages (vr, vj) ===

# 1. Calculate initial guesses based on polar coordinates (Vm, Va)
vr_init = {row.Index: row.Vm * math.cos(math.radians(row.Va)) for row in case['bus'].itertuples()}
vj_init = {row.Index: row.Vm * math.sin(math.radians(row.Va)) for row in case['bus'].itertuples()}

# 2. Define the bounds rule: Both vr and vj are bounded by (-Vmax, Vmax)
def vr_vj_bounds_rule(m, b):
    vmax = case['bus'].at[b, 'Vmax']
    return (-vmax, vmax)

# 3. Declare the variables natively on the model
model.vr = pyo.Var(model.Buses, domain=pyo.Reals, initialize=vr_init, bounds=vr_vj_bounds_rule)
model.vj = pyo.Var(model.Buses, domain=pyo.Reals, initialize=vj_init, bounds=vr_vj_bounds_rule)

# === RSV Step 2: Reference Bus Angle Fixing ===

# 1. Identify the Reference Bus 
# (In PGLib/MATPOWER, 'type' == 3 indicates the slack/reference bus)
ref_bus_row = case['bus'][case['bus']['type'] == 3]
ref_bus = ref_bus_row.index[0]

# 2. Get the reference angle in radians
ref_angle_deg = float(ref_bus_row['Va'].iloc[0])
ref_angle_rad = math.radians(ref_angle_deg)

# 3. Apply the Reference Bus logic
if ref_angle_deg == 0.0:
    # Standard Case: Angle is 0
    model.vj[ref_bus].fix(0.0)
    
    # Since Vj is 0, Vr is the full voltage magnitude, so it must be >= Vmin
    v_min = case['bus'].at[ref_bus, 'Vmin']
    model.vr[ref_bus].setlb(v_min)
    
else:
    # Rare Case: Reference angle is NOT 0
    # We enforce tan(theta) = Vj / Vr  =>  Vj = Vr * tan(theta)
    tan_theta = math.tan(ref_angle_rad)
    
    model.eq_ref_bus_nonzero = pyo.Constraint(
        expr=model.vj[ref_bus] == tan_theta * model.vr[ref_bus]
    )
    
# === RSV Step 3: Voltage Magnitude Squared Link ===
# Enforce the Pythagorean relationship: V^2 = Vr^2 + Vj^2

# Initialize the equality constraint on the model
model.eq_vmsq = pyo.Constraint(model.Buses)

for b in model.Buses:
    # Set vmsq equal to the sum of the squares of the rectangular components
    model.eq_vmsq[b] = model.vmsq[b] == model.vr[b]**2 + model.vj[b]**2

# === RSV Step 4: Link 'c' Variable to Rectangular Voltages ===

# Initialize the equality constraint on the UniqueBusPairs set
model.eq_c = pyo.Constraint(model.UniqueBusPairs)

for i, j in model.UniqueBusPairs:
    # Enforce c_ij = (Vr_i * Vr_j) + (Vj_i * Vj_j)
    model.eq_c[(i, j)] = (
        model.c[(i, j)] == model.vr[i] * model.vr[j] + model.vj[i] * model.vj[j]
    )    

# === RSV Step 5: Link 's' Variable to Rectangular Voltages ===

# Initialize the equality constraint on the UniqueBusPairs set
model.eq_s = pyo.Constraint(model.UniqueBusPairs)

for i, j in model.UniqueBusPairs:
    # Enforce s_ij = (Vj_i * Vr_j) - (Vr_i * Vj_j)
    model.eq_s[(i, j)] = (
        model.s[(i, j)] == model.vj[i] * model.vr[j] - model.vr[i] * model.vj[j]
    )    

In [11]:
# 1. Execute the solve 
# (Pyomo automatically loads the solutions back into the model by default!)
model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
solver = pyo.SolverFactory('ipopt')
results = solver.solve(model, tee=True)

# 2. Verify the Termination Condition
term_cond = results.solver.termination_condition

if term_cond == pyo.TerminationCondition.optimal or term_cond == pyo.TerminationCondition.locallyOptimal:
    print("\n--- OPTIMAL SOLUTION FOUND ---")
elif term_cond == pyo.TerminationCondition.infeasible:
    raise Exception("\n--- MODEL IS INFEASIBLE ---")
else:
    raise Exception(f"Problem encountered during solve! Condition: {term_cond}")

# ==========================================
# 20. Extract Results into DataFrames
# ==========================================

# 1. Total Objective Cost
case['total_cost'] = pyo.value(model.obj)

# # 2. Extract Generator Dispatch
# # Index 1 = PG, Index 2 = QG
# case['gen'].iloc[:, 1] = [pyo.value(model.pg[g]) for g in model.Gens]
# case['gen'].iloc[:, 2] = [pyo.value(model.qg[g]) for g in model.Gens]

# (Optional) Keep your MW columns for your own manual reading
case['gen']['Pg_MW'] = case['gen'].iloc[:, 1] * baseMVA
case['gen']['Qg_MVAr'] = case['gen'].iloc[:, 2] * baseMVA

# 3. Extract Bus Voltages and LMPs (Shadow Prices)
lmp_list = []
qlmp_list = []
pl_list = []
vm_list = []
va_list = []

for b in model.Buses:
    # Shadow Prices (LMPs)
    lmp_list.append(pyo.value(model.dual[model.eq_p_balance[b]]))
    qlmp_list.append(pyo.value(model.dual[model.eq_q_balance[b]]))
    
    # Active Power Load
    pl_list.append(pyo.value(model.pl[b]))
    
    # Rectangular to Polar (Vm, Va)
    vr_val = pyo.value(model.vr[b])
    vj_val = pyo.value(model.vj[b])
    
    vm_list.append(math.sqrt(vr_val**2 + vj_val**2))
    va_list.append(math.degrees(math.atan2(vj_val, vr_val)))

# Assign to Bus DataFrame
# Check if columns exist; if not, you might need to pad the matrix
# if case['bus'].shape[1] < 15:
#     # Ensure matrix is wide enough for shadow prices
#     new_cols = np.zeros((case['bus'].shape[0], 15 - case['bus'].shape[1]))
#     case['bus'] = pd.concat([case['bus'], pd.DataFrame(new_cols)], axis=1)

# case['bus'].iloc[:, 13] = lmp_list  # Shadow Price P
# case['bus'].iloc[:, 14] = qlmp_list # Shadow Price Q
# case['bus']['pl_optimal'] = pl_list
# case['bus'].iloc[:, 7] = vm_list
# case['bus'].iloc[:, 8] = va_list

# 4. Extract Branch Power Flows
# Ensure matrix is wide enough for results (17 columns total)
if case['branch'].shape[1] < 17:
    new_branch_cols = np.zeros((case['branch'].shape[0], 17 - case['branch'].shape[1]))
    case['branch'] = pd.concat([case['branch'], pd.DataFrame(new_branch_cols)], axis=1)

# Index 13=PF, 14=QF, 15=PT, 16=QT
case['branch'].iloc[:, 13] = [pyo.value(model.pf[k]) for k in model.Branches]
case['branch'].iloc[:, 14] = [pyo.value(model.qf[k]) for k in model.Branches]
case['branch'].iloc[:, 15] = [pyo.value(model.pt[k]) for k in model.Branches]
case['branch'].iloc[:, 16] = [pyo.value(model.qt[k]) for k in model.Branches]

print(f"Extraction Complete! Total Cost: {case['total_cost']:,.2f}")


Ipopt 3.11.1: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

NOTE: You are using Ipopt by default with the MUMPS linear solver.
      Other linear solvers might be more efficient (see Ipopt documentation).


This is Ipopt version 3.11.1, running with linear solver mumps.

Number of nonzeros in equality constraint Jacobian...:    12990
Number of nonzeros in inequality constraint Jacobian.:     3288
Number of nonzeros in Lagrangian Hessian.............:     3877

Total number of variables............................:     3499
                     variables with only lower bounds:        0
                variables with lower and upper bounds:     3499


In [12]:
# === 1. Configuration ===
Ybus_matrix,_, _ = makeYbus(baseMVA, case['bus'], case['branch'])
match = re.search(r'case(\d+)', case_name)
seed = 2026
if match:
    nbus = int(match.group(1))
max_change_load = 0.1
# Ensure you have your 'case' dictionary (DataFrames) and 'model' ready

# Identify load buses (where Pd > 0 or Qd > 0)
load_bus_indices = case['bus'][(case['bus']['Pd'] != 0) | (case['bus']['Qd'] != 0)].index.tolist()
NL = len(load_bus_indices)

# === 2. Initialize Storage (To match MATLAB format) ===
# We store as (nsamples, nbus) to skip the transpose step later
all_dem = np.zeros((nsamples, nbus), dtype=complex)
all_gen = np.zeros((nsamples, len(model.Gens)), dtype=complex)
all_vol = np.zeros((nsamples, nbus), dtype=complex)

# === 3. Scenario Generation Loop ===
print(f"Generating {nsamples} scenarios using Pyomo + Ipopt...")

for s in range(nsamples):
    # Match MATLAB perturbation: (rand * 0.2) + 0.9
    load_factor_p = np.random.uniform(1 - max_change_load, 1 + max_change_load, size=NL)
    load_factor_q = np.random.uniform(1 - max_change_load, 1 + max_change_load, size=NL)

    # Update Pyomo model fixed loads
    for i, bus_idx in enumerate(load_bus_indices):
        new_p = case['bus'].at[bus_idx, 'Pd'] * load_factor_p[i]
        new_q = case['bus'].at[bus_idx, 'Qd'] * load_factor_q[i]
        
        model.pl[bus_idx].fix(new_p)
        model.ql[bus_idx].fix(new_q)

    # Solve
    solver = pyo.SolverFactory('ipopt')
    results_obj = solver.solve(model, tee=False)

    if results_obj.solver.termination_condition == pyo.TerminationCondition.optimal:
        # Extract Results in Complex Format (P + jQ)
        # 1. Demand (Convert p.u. -> MW/MVAr)
        for i, b in enumerate(model.Buses):
            p_mw = pyo.value(model.pl[b]) * baseMVA
            q_mvar = pyo.value(model.ql[b]) * baseMVA
            all_dem[s, i] = p_mw + 1j * q_mvar
        
        # 2. Generation (Convert p.u. -> MW/MVAr)
        for i, g in enumerate(model.Gens):
            pg_mw = pyo.value(model.pg[g]) * baseMVA
            qg_mvar = pyo.value(model.qg[g]) * baseMVA
            all_gen[s, i] = pg_mw + 1j * qg_mvar
            
        # 3. Voltage (Vr + jVj) 
        for i, b in enumerate(model.Buses):
            all_vol[s, i] = pyo.value(model.vr[b]) + 1j * pyo.value(model.vj[b])
            
        print(f"  Scenario {s+1}/{nsamples} Success")
    else:
        print(f"  Scenario {s+1}/{nsamples} Failed to converge")

# === 4. Bundle into the final Pickle structure ===
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

# 1. Skip the first column (Branch ID) using [:, 1:]
branch_data = case['branch'].values[:, 1:].copy()
bus_data = case['bus'].values.copy()
cols = np.r_[0, 2:11]
gen_data = case['gen'].values[:,cols].copy()
gencost_data = case['gencost'].values[:,2:].copy()

final_data = {
    'Dem': all_dem,
    'Gen': all_gen,
    'Vol': all_vol,
    'ppc': {
        'bus': bus_data,
        'gen': gen_data,
        'branch': branch_data, # Now has 13 columns
        'gencost': gencost_data,
        'baseMVA': float(case['baseMVA']['baseMVA'][0]),
        'version': 2
    },
    'Ybus': Ybus_matrix,
    'EPS_INTERIOR' : np.array([[0]],dtype=np.uint8), # np.array([[1e-4]])
    'MaxChangeLoad': np.array([[max_change_load]]),
    'CorrCoeff': np.array([[0.5]]),
    '__globals__': [],
    '__version__': '1.0',
    '__header__': b'Pyomo, Platform: MSI, Created on: 2026'
}

# Save
filename = f"datasets/acopf/acopf_{seed}_{nbus}_{nsamples}_dataset"
with open(filename, 'wb') as f:
    pickle.dump(final_data, f)

print(f"Dataset saved as: {filename}")


Generating 9 scenarios using Pyomo + Ipopt...
  Scenario 1/9 Success
  Scenario 2/9 Success
  Scenario 3/9 Success
  Scenario 4/9 Success
  Scenario 5/9 Success
  Scenario 6/9 Success
  Scenario 7/9 Success
  Scenario 8/9 Success
  Scenario 9/9 Success
Dataset saved as: datasets/acopf/acopf_2026_300_9_dataset


In [8]:
# seed = 2026
# nbus = 30
# nsamples = 10
# filepath = f'datasets\\acopf\\acopf_{seed}_{nbus}_{nsamples}_dataset'
# with open(filepath, 'rb') as f:
#     dataset = pickle.load(f)
# dataset['ppc'].keys() # 

In [9]:
print('bismillah')

bismillah
